In [1]:
import chromadb
import sentence_transformers
import anthropic

print("All libraries imported successfully!")

All libraries imported successfully!


In [2]:
import sys
!{sys.executable} -m pip install chromadb sentence-transformers anthropic

In [3]:
import chromadb
import sentence_transformers
import anthropic

print("All libraries imported successfully!")

All libraries imported successfully!


In [4]:
documents = [
    "The Eiffel Tower is located in Paris, France. It was built in 1889 and stands 330 meters tall.",
    "The Great Wall of China stretches over 21,000 kilometers. It was built to protect Chinese states from invasions.",
    "The Colosseum is an ancient amphitheater in Rome, Italy. It was completed in 80 AD and could hold up to 80,000 spectators.",
    "The Taj Mahal is a white marble mausoleum in Agra, India. It was built by Emperor Shah Jahan in memory of his wife.",
    "The Statue of Liberty is located in New York Harbor, USA. It was a gift from France to the United States in 1886.",
]

print(f"We have {len(documents)} documents in our knowledge base.")

We have 5 documents in our knowledge base.


In [5]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(documents)

print(f"Each document was turned into {len(embeddings[0])} numbers")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Each document was turned into 384 numbers


In [6]:
import chromadb

client = chromadb.Client()
collection = client.create_collection("landmarks")

collection.add(
    documents=documents,
    embeddings=embeddings.tolist(),
    ids=["doc1", "doc2", "doc3", "doc4", "doc5"]
)

print(f"Stored {collection.count()} documents in ChromaDB!")

Stored 5 documents in ChromaDB!


In [7]:
question = "Where is the Eiffel Tower?"

question_embedding = model.encode([question])

results = collection.query(
    query_embeddings=question_embedding.tolist(),
    n_results=2
)

print("Question:", question)
print("\nMost relevant documents found:")
for doc in results['documents'][0]:
    print("-", doc)

Question: Where is the Eiffel Tower?

Most relevant documents found:
- The Eiffel Tower is located in Paris, France. It was built in 1889 and stands 330 meters tall.
- The Statue of Liberty is located in New York Harbor, USA. It was a gift from France to the United States in 1886.


In [10]:
import anthropic
question = "Tell me about the Colosseum"
client = anthropic.Anthropic(api_key=""your-api-key-here")

retrieved_docs = results['documents'][0]
context = "\n".join(retrieved_docs)

message = client.messages.create(
    model="claude-opus-4-6",
    max_tokens=1024,
    messages=[
        {
            "role": "user",
            "content": f"""Use only the information below to answer the question.

Context:
{context}

Question: {question}"""
        }
    ]
)

print("Claude's answer:")
print(message.content[0].text)

Claude's answer:
Based on the information provided, I cannot answer your question about the Colosseum. The context only contains information about the Eiffel Tower and the Statue of Liberty. There is no mention of the Colosseum in the given text.
